# L9 — VWAP Dinámico + Cierre de Ciclo: Ejercicios

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | 1–5 | Completar en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o alumnos adelantados |

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'monospace', 'axes.facecolor': '#18181b',
                     'figure.facecolor': '#09090b', 'axes.edgecolor': '#27272a',
                     'grid.color': '#27272a', 'text.color': '#e4e4e7'})

CYAN, GREEN, RED, AMBER, MUTED = '#22d3ee', '#4ade80', '#f87171', '#f59e0b', '#a1a1aa'
TOTAL_QTY = 10.0
WINDOW    = 5
TEST_DATE = '2025-12-21'

---
## Ejercicio 0 — Reflexión (sin código)

Lee este escenario y responde las preguntas:

In [ ]:
print("""ESCENARIO:
Son las 02:00 UTC. Tu algoritmo necesita comprar 10 BTC a lo largo del día.
El schedule estático predecía ejecutar 0.048 BTC en el intervalo 0 (00:00).
El mercado ejecutó 0.096 BTC en ese intervalo — el doble de lo previsto.

Preguntas:
1. ¿Cuál es el correction factor (CF) para ese bloque?
2. Si el schedule restante era de 9.90 BTC, ¿cómo lo ajustas con el CF?
3. ¿Por qué puede ser arriesgado multiplicar SIEMPRE por el CF del último bloque?
   (Pista: piensa en el ruido del mercado)
4. ¿Qué tipo de señal del LOB (de L6/L7) podría confirmar que el mercado
   será activo durante todo el día y no solo en ese bloque?
""")

---
## Ejercicio 1 — Núcleo: Cargar datos y separar train/test

In [ ]:
# Carga '../08-vwap-volume-baselines/data/btc_volume_intraday.csv'
# Separa train_df (dates < TEST_DATE) y test_df (date == TEST_DATE)
# Calcula static_profile: media de volume_normalized por interval_idx en train_df
# Extrae actual_day21: array de volume_normalized del día de test, ordenado por interval_idx

df          = None  # TODO
train_df    = None  # TODO
test_df     = None  # TODO
static_profile = None  # TODO
actual_day21   = None  # TODO

print(f'Train: {train_df["date"].nunique() if train_df is not None else "?"} días')
print(f'Test:  {test_df["date"].nunique() if test_df is not None else "?"} día')

In [ ]:
# Validador E1
assert df is not None and len(df) == 6048, "df debe tener 6048 filas"
assert len(train_df) == len(df[df['date'] < TEST_DATE]), "train_df: todos los días antes del test"
assert len(test_df) == 288, f"test_df debe tener 288 intervalos, tienes {len(test_df)}"
assert len(static_profile) == 288, "static_profile debe tener 288 valores"
assert len(actual_day21) == 288, "actual_day21 debe tener 288 valores"
assert abs(static_profile.iloc[0] - 0.00476613) < 1e-5, \
    f"static_profile[0] debe ser ~0.004766 (media de 20 días). Tienes {static_profile.iloc[0]:.6f}"
print(f"✓ E1 correcto — train={train_df['date'].nunique()} días, test={TEST_DATE}")

In [ ]:
# Solución E1
df = pd.read_csv('../08-vwap-volume-baselines/data/btc_volume_intraday.csv', parse_dates=['datetime'])
train_df = df[df['date'] < TEST_DATE]
test_df  = df[df['date'] == TEST_DATE]
static_profile = train_df.groupby('interval_idx')['volume_normalized'].mean()
actual_day21   = test_df.sort_values('interval_idx')['volume_normalized'].values
print(f'Train: {train_df["date"].nunique()} días  |  Test: {TEST_DATE}  |  Profile sum: {static_profile.sum():.6f}')

---
## Ejercicio 2 — Núcleo: `compute_correction_factor`

In [ ]:
# Implementa compute_correction_factor(realized_block, predicted_block)
# - Retorna realized_block.sum() / predicted_block.sum()
# - Si predicted_block.sum() == 0, retorna 1.0
#
# Luego calcula cf_block_0: CF del bloque 0 del día 21 (intervalos 0-4)

def compute_correction_factor(realized_block: np.ndarray,
                               predicted_block: np.ndarray) -> float:
    pass  # TODO

cf_block_0 = None  # TODO

print(f'CF bloque 0: {cf_block_0}')

In [ ]:
# Validador E2
assert callable(compute_correction_factor), "compute_correction_factor debe ser una función"
assert abs(cf_block_0 - 0.988095) < 1e-4, \
    f"CF bloque 0 debe ser ~0.9881. Tienes {cf_block_0:.6f}"
# Edge case: predicted = 0
assert compute_correction_factor(np.array([0.1]), np.array([0.0])) == 1.0, \
    "Si predicted_sum==0, debe retornar 1.0"
# CF > 1
assert compute_correction_factor(np.array([2.0]), np.array([1.0])) == 2.0
print(f"✓ E2 correcto — CF bloque 0: {cf_block_0:.6f} (mercado {cf_block_0:.3f}× predicho)")

In [ ]:
# Solución E2
def compute_correction_factor(realized_block, predicted_block):
    pred_sum = predicted_block.sum()
    if pred_sum == 0:
        return 1.0
    return float(realized_block.sum() / pred_sum)

cf_block_0 = compute_correction_factor(actual_day21[0:5], static_profile.values[0:5])
print(f'CF bloque 0: {cf_block_0:.6f}')
print(f'  Predicho:  {static_profile.values[0:5].sum():.6f}')
print(f'  Realizado: {actual_day21[0:5].sum():.6f}')

---
## Ejercicio 3 — Núcleo: Tracking deviation estática

In [ ]:
# Calcula la tracking deviation del schedule ESTÁTICO en el día 21:
#   cum_target  = np.cumsum(actual_day21) * TOTAL_QTY
#   cum_static  = np.cumsum(static_profile.values) * TOTAL_QTY
#   dev_static  = abs diferencia punto a punto
#   max_dev_static = máximo de dev_static

cum_target     = None  # TODO
cum_static     = None  # TODO
dev_static     = None  # TODO
max_dev_static = None  # TODO

print(f'Max tracking deviation estática: {max_dev_static}')

In [ ]:
# Validador E3
assert max_dev_static is not None, "Calcula max_dev_static"
assert abs(max_dev_static - 0.126737) < 0.01, \
    f"max_dev_static debe ser ~0.1267 BTC. Tienes {max_dev_static:.6f}"
assert len(dev_static) == 288, "dev_static debe tener 288 valores"
print(f"✓ E3 correcto — max tracking deviation estática: {max_dev_static:.4f} BTC")

In [ ]:
# Solución E3
cum_target  = np.cumsum(actual_day21) * TOTAL_QTY
cum_static  = np.cumsum(static_profile.values) * TOTAL_QTY
dev_static  = np.abs(cum_static - cum_target)
max_dev_static = dev_static.max()

print(f'Máxima desviación estática: {max_dev_static:.4f} BTC')
print(f'En qué intervalo: {dev_static.argmax()} ({dev_static.argmax()*5//60:02d}:{dev_static.argmax()*5%60:02d} UTC)')

---
## Ejercicio 4 — Núcleo: `walk_forward_dynamic` y tracking dinámica

In [ ]:
# Implementa walk_forward_dynamic(actual, base_profile, window=5):
#   Para cada bloque de `window` intervalos:
#     1. Ejecuta el schedule actual para esos intervalos
#     2. Si quedan intervalos, calcula CF y multiplica el schedule restante
#   Retorna: (dynamic_schedule array, cf_history list)
#
# Luego calcula max_dev_dynamic usando la misma lógica que E3

def walk_forward_dynamic(actual: np.ndarray, base_profile: np.ndarray,
                          window: int = 5) -> tuple:
    pass  # TODO

dyn_sched, cf_hist = None, None  # TODO
max_dev_dynamic    = None        # TODO

print(f'Max tracking deviation dinámica: {max_dev_dynamic}')

In [ ]:
# Validador E4
assert callable(walk_forward_dynamic), "walk_forward_dynamic debe ser una función"
assert dyn_sched is not None and len(dyn_sched) == 288, "dyn_sched debe tener 288 valores"
assert cf_hist is not None and len(cf_hist) == 57, \
    f"cf_hist debe tener 57 valores (bloques 0-56, el último no genera CF). Tienes {len(cf_hist) if cf_hist else '?'}"
assert max_dev_dynamic is not None
assert abs(max_dev_dynamic - 0.027309) < 0.01, \
    f"max_dev_dynamic debe ser ~0.0273 BTC. Tienes {max_dev_dynamic:.6f}"
assert max_dev_dynamic < max_dev_static, \
    "El dinámico debe tener menor tracking deviation que el estático"

improvement = (max_dev_static - max_dev_dynamic) / max_dev_static * 100
print(f"✓ E4 correcto — dinámica: {max_dev_dynamic:.4f} BTC (mejora {improvement:.0f}% vs estático)")

In [ ]:
# Solución E4
def walk_forward_dynamic(actual, base_profile, window=5):
    n = len(actual)
    schedule = base_profile.copy()
    dynamic_schedule = np.zeros(n)
    cf_history = []
    for j in range(0, n, window):
        dynamic_schedule[j:j+window] = schedule[j:j+window]
        if j + window < n:
            cf = compute_correction_factor(actual[j:j+window], schedule[j:j+window])
            cf_history.append(cf)
            schedule[j+window:] *= cf
    return dynamic_schedule, cf_history

dyn_sched, cf_hist = walk_forward_dynamic(actual_day21, static_profile.values)
cum_dynamic = np.cumsum(dyn_sched / dyn_sched.sum()) * TOTAL_QTY
max_dev_dynamic = np.abs(cum_dynamic - cum_target).max()

print(f'Estático:  {max_dev_static:.4f} BTC  →  Dinámico: {max_dev_dynamic:.4f} BTC')
print(f'Mejora: {(max_dev_static-max_dev_dynamic)/max_dev_static*100:.1f}%')

---
## Ejercicio 5 — Núcleo: Visualización de tracking deviation

In [ ]:
# Crea una figura con dos fill_between:
#   - dev_static * 1000 en rojo (mBTC)
#   - dev_dynamic * 1000 en verde (mBTC)
# Añade etiquetas de hora en x: 00:00, 06:00, 12:00, 18:00, 23:55
# Título con la mejora porcentual

dev_dynamic = np.abs(cum_dynamic - cum_target)

fig, ax = plt.subplots(figsize=(12, 4))

# TODO: fill_between, xticks, labels, title, legend, grid

plt.tight_layout()
plt.show()

In [ ]:
# Validador E5 (light)
assert plt.get_fignums(), "Debes crear una figura"
print("✓ E5 — figura creada. Verifica que:")
print("  1. El área verde (dinámico) es claramente más pequeña que la roja (estático)")
print(f"  2. El pico rojo (~127 mBTC) es visible")
print(f"  3. El pico verde (~27 mBTC) es visible")

In [ ]:
# Solución E5
dev_dynamic = np.abs(cum_dynamic - cum_target)
improvement = (max_dev_static - max_dev_dynamic) / max_dev_static * 100

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(range(288), dev_static*1000, alpha=0.35, color=RED, label=f'Estático ({max_dev_static*1000:.0f} mBTC)')
ax.fill_between(range(288), dev_dynamic*1000, alpha=0.6, color=GREEN, label=f'Dinámico ({max_dev_dynamic*1000:.0f} mBTC)')
ax.set_xticks([0, 72, 144, 216, 287])
ax.set_xticklabels(['00:00', '06:00', '12:00', '18:00', '23:55'])
ax.set_xlabel('Hora (UTC)', color=MUTED)
ax.set_ylabel('Tracking deviation (mBTC)', color=MUTED)
ax.set_title(f'Tracking deviation — mejora {improvement:.0f}%', color=CYAN, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Ejercicio 6 — Si vamos bien: Walk-forward backtest completo

In [ ]:
# Evalúa ambos modelos en walk-forward sobre días 6-21 (16 días).
# Para cada día de test:
#   - train = todos los días anteriores
#   - static_profile = media de train
#   - Calcula max_dev_static y max_dev_dynamic
# Guarda en backtest_df (DataFrame): 'date', 'static', 'dynamic', 'dyn_wins'
# Calcula:
#   days_dynamic_wins: cuántos días gana el dinámico (int)
#   mean_improvement_pct: mejora media porcentual (float)

backtest_df       = None  # TODO
days_dynamic_wins = None  # TODO
mean_improvement_pct = None  # TODO

print(f'Días donde gana dinámico: {days_dynamic_wins}')
print(f'Mejora media: {mean_improvement_pct:.1f}%')

In [ ]:
# Validador E6
assert backtest_df is not None and len(backtest_df) == 16, \
    f"backtest_df debe tener 16 filas (días 6-21). Tienes {len(backtest_df) if backtest_df is not None else '?'}"
assert days_dynamic_wins >= 13, \
    f"El dinámico debe ganar en al menos 13 de 16 días. Ganó en {days_dynamic_wins}"
assert 35 < mean_improvement_pct < 60, \
    f"La mejora media debe estar entre 35% y 60%. Tienes {mean_improvement_pct:.1f}%"
print(f"✓ E6 correcto — dinámico gana {days_dynamic_wins}/16 días, mejora media {mean_improvement_pct:.1f}%")

In [ ]:
# Solución E6
dates_sorted = sorted(df['date'].unique())
results = []
for i, test_date in enumerate(dates_sorted):
    if i < 5:
        continue
    train_d = df[df['date'] < test_date]
    test_d  = df[df['date'] == test_date]
    sp = train_d.groupby('interval_idx')['volume_normalized'].mean().values
    act = test_d.sort_values('interval_idx')['volume_normalized'].values
    dyn, _ = walk_forward_dynamic(act, sp.copy())
    ct = np.cumsum(act) * TOTAL_QTY
    cs = np.cumsum(sp) * TOTAL_QTY
    cd = np.cumsum(dyn / dyn.sum()) * TOTAL_QTY
    mds = np.abs(cs - ct).max()
    mdd = np.abs(cd - ct).max()
    results.append({'date': test_date, 'static': mds, 'dynamic': mdd, 'dyn_wins': mdd < mds})

backtest_df = pd.DataFrame(results)
days_dynamic_wins = int(backtest_df['dyn_wins'].sum())
mean_improvement_pct = float((backtest_df['static'] - backtest_df['dynamic']) / backtest_df['static'] * 100).mean()

print(f'Días dinámico gana: {days_dynamic_wins}/16')
print(f'Mejora media: {mean_improvement_pct:.1f}%')

---
## Ejercicio 7 — Si vamos bien: `ExecutionDecision`

In [ ]:
# Implementa la clase ExecutionDecision con:
#   - __init__(self, imbalance, fill_prob, volume_ratio)
#   - decide(self) → 'LIMIT', 'MARKET' o 'WAIT'
#
# Lógica:
#   LIMIT:  imbalance > 0.55 AND fill_prob > 0.50
#   MARKET: volume_ratio > 1.20 AND (imbalance < 0.45 OR fill_prob <= 0.50)
#   WAIT:   cualquier otro caso

class ExecutionDecision:
    IMBALANCE_BULL = 0.55
    FILL_MIN       = 0.50
    VOL_HIGH       = 1.20
    
    def __init__(self, imbalance: float, fill_prob: float, volume_ratio: float):
        pass  # TODO
    
    def decide(self) -> str:
        pass  # TODO

# Test rápido
print(ExecutionDecision(0.62, 0.65, 1.35).decide())  # debe ser LIMIT
print(ExecutionDecision(0.38, 0.35, 1.55).decide())  # debe ser MARKET
print(ExecutionDecision(0.50, 0.45, 0.65).decide())  # debe ser WAIT

In [ ]:
# Validador E7
assert ExecutionDecision(0.62, 0.65, 1.35).decide() == 'LIMIT', \
    "(0.62, 0.65, 1.35) debe ser LIMIT — imbalance alto + fill alto"
assert ExecutionDecision(0.38, 0.35, 1.55).decide() == 'MARKET', \
    "(0.38, 0.35, 1.55) debe ser MARKET — vol alto + no alcista + fill bajo"
assert ExecutionDecision(0.50, 0.45, 0.65).decide() == 'WAIT', \
    "(0.50, 0.45, 0.65) debe ser WAIT — neutro y vol bajo"
assert ExecutionDecision(0.60, 0.51, 0.90).decide() == 'LIMIT', \
    "(0.60, 0.51, 0.90) debe ser LIMIT — imbalance > 0.55 y fill > 0.50"

# Edge: justo en el umbral
assert ExecutionDecision(0.55, 0.50, 1.0).decide() == 'WAIT', \
    "Con imbalance==0.55 y fill==0.50, los umbrales son estrictos (>), debe ser WAIT"

print("✓ E7 correcto — ExecutionDecision implementada correctamente")

In [ ]:
# Solución E7
class ExecutionDecision:
    IMBALANCE_BULL = 0.55
    FILL_MIN       = 0.50
    VOL_HIGH       = 1.20
    
    def __init__(self, imbalance, fill_prob, volume_ratio):
        self.imbalance    = imbalance
        self.fill_prob    = fill_prob
        self.volume_ratio = volume_ratio
    
    def decide(self):
        if self.imbalance > self.IMBALANCE_BULL and self.fill_prob > self.FILL_MIN:
            return 'LIMIT'
        elif (self.volume_ratio > self.VOL_HIGH and
              (self.imbalance < (1 - self.IMBALANCE_BULL) or self.fill_prob <= self.FILL_MIN)):
            return 'MARKET'
        else:
            return 'WAIT'

for imb, fp, vr, expected in [(0.62,0.65,1.35,'LIMIT'),(0.38,0.35,1.55,'MARKET'),(0.50,0.45,0.65,'WAIT')]:
    got = ExecutionDecision(imb, fp, vr).decide()
    status = '✓' if got == expected else '✗'
    print(f'{status} ({imb},{fp},{vr}) → {got}')

---
## Ejercicio 8 — Bonus: Scan `ExecutionDecision` sobre LOB real

In [ ]:
# Carga '../07-lob-modeling-examples/data/lob_modeling_features.csv'
# Para cada fila, crea un ExecutionDecision con:
#   imbalance    = row['imbalance_mean_5']
#   fill_prob    = 0.5 + 0.2 * (2 * row['fill_in_3'] - 1)  # proxy
#   volume_ratio = 1.0  # sin señal de volumen
# Guarda las decisiones en una lista `decisions`
# Computa decision_counts: pd.Series con .value_counts()

lob_df         = None  # TODO
decisions      = []    # TODO
decision_counts = None # TODO

print(decision_counts)

In [ ]:
# Validador E8
assert len(decisions) == 484, f"Debe haber 484 decisiones. Tienes {len(decisions)}"
assert set(decisions).issubset({'LIMIT', 'MARKET', 'WAIT'}), \
    "Solo se permiten los valores 'LIMIT', 'MARKET', 'WAIT'"
assert 'WAIT' in decision_counts.index, "Debe haber al menos una decisión WAIT"
assert decision_counts.sum() == 484
print(f"✓ E8 correcto — {len(decisions)} decisiones analizadas")
print(decision_counts.to_string())

In [ ]:
# Solución E8
lob_df = pd.read_csv('../07-lob-modeling-examples/data/lob_modeling_features.csv')
decisions = []
for _, row in lob_df.iterrows():
    fill_proxy = 0.5 + 0.2 * (2 * row['fill_in_3'] - 1)
    d = ExecutionDecision(row['imbalance_mean_5'], fill_proxy, 1.0)
    decisions.append(d.decide())
decision_counts = pd.Series(decisions).value_counts()
print(decision_counts)
print(f'\nDistribución: {dict(decision_counts/len(decisions)*100)}')

---
## Ejercicio 9 — Bonus: Panel visual del cuadro de mando

In [ ]:
# Crea una figura 1×3 subplots mostrando para el día 21:
#   1. Subplot izquierdo: barras de CF por bloque (coloreadas verde/rojo según CF>1)
#   2. Subplot central: área de tracking deviation estática vs dinámica
#   3. Subplot derecho: pie chart de distribución de decisiones del E8
#
# Usa el esquema de colores del curso (CYAN, GREEN, RED, AMBER, MUTED)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# TODO: rellenar los 3 subplots

fig.suptitle('Cuadro de Mando — L9 Cierre de Ciclo', color=CYAN, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Validador E9 (light)
assert plt.get_fignums(), "Debes crear una figura con 3 subplots"
print("✓ E9 — figura del cuadro de mando creada")
print("  Verifica que los 3 subplots muestran información distinta y coherente")

---
## Ejercicio 10 — Bonus: Repaso integral L4→L9 (prep examen)

In [ ]:
# Completa la tabla de conceptos del bloque para preparar el Exam-Quiz I:
print("""
TABLA DE CONCEPTOS BLOQUE 1 (L4–L9)
====================================

Para cada clase, escribe con tus palabras:
  A) La pregunta central que responde
  B) El artefacto de código más importante
  C) La limitación principal que motivó la siguiente clase

L4: LOB Data — 
  A) ?
  B) ?
  C) ?

L5: Order Types — 
  A) ?
  B) ?
  C) ?

L6: LOB Pipeline — 
  A) ?
  B) ?
  C) ?

L7: LOB Modeling — 
  A) ?
  B) ?
  C) ?

L8: VWAP Baselines — 
  A) ?
  B) ?
  C) ?

L9: VWAP Dynamic — 
  A) ?
  B) ?
  C) ?
""")

In [ ]:
# Solución E10 (referencia — respuestas posibles)
print("""
L4: LOB Data
  A) ¿Cómo se estructura el libro de órdenes y qué información contiene?
  B) btc_lob_snapshots.csv + features mid, spread, imbalance
  C) Tenemos datos pero no podemos predecir nada con snapshot estático

L5: Order Types
  A) ¿Cuándo se ejecuta una orden límite en el LOB?
  B) Fill probability empírica por nivel de precio
  C) La ejecución es probabilística — necesitamos modelarla

L6: LOB Pipeline
  A) ¿Puedes predecir la dirección del precio con el LOB?
  B) LogisticRegression + pipeline completo (split temporal, leakage check)
  C) 49.3% de accuracy — peor que una moneda sin temporal features

L7: LOB Modeling
  A) ¿Mejoran los features temporales? ¿Qué modelo funciona mejor?
  B) walk_forward_temporal, DecisionTree (overfitting demo), RandomForest + ExecutionDecision
  C) 55.5% es micro-escala — no sabe cuándo hay liquidez para ejecutar en escala grande

L8: VWAP Baselines
  A) ¿Cuándo hay liquidez intradiaria? ¿Cómo construyo un schedule?
  B) build_vwap_schedule(total_qty, volume_profile) + rmse_profile()
  C) El schedule es estático — si el día es diferente al histórico, te descuadras

L9: VWAP Dynamic
  A) ¿Cómo adapto el schedule en tiempo real?
  B) walk_forward_dynamic() + ExecutionDecision
  C) Datos sintéticos cortos — en producción necesitas meses de historia y validación OOS
""")